# H100x8 GPU-only vs Hybrid Comparison

이 노트북은 `eval/basic/H100x8` 하위 결과를 자동으로 스캔해서 GPU-only 기준선과 hybrid 실행들을 비교합니다.

분석 기준:

- 결과 JSON: `gpu_only.json`, `hybrid.json`
- 설정 JSON: `system_info.json`
- 로그: `hybrid_server_boot.log`, `hybrid_server_run.log`
- `inspect.txt` 같은 후처리 산출물은 사용하지 않습니다.

주요 질문:

- GPU-only 대비 hybrid wall time / throughput이 얼마나 악화 또는 개선되는가
- CPU가 실제로 몇 개 request를 가져갔는가
- `cpu_max_num_seqs=1`과 `16`의 차이가 tail로 어떻게 드러나는가
- CPU pinning만 달라졌는지, 실행 패턴도 달라졌는지


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
from pathlib import Path

import pandas as pd
try:
    from IPython.display import display, Markdown
except ModuleNotFoundError:
    def display(obj):
        print(obj)
    def Markdown(text):
        return text

try:
    os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

ROOT = Path.cwd()
if ROOT.name != "H100x8":
    ROOT = Path("/vllm_hybrid/eval/basic/H100x8")

assert ROOT.exists(), ROOT
ROOT


## Parsing Helpers

In [ ]:
ANSI_RE = re.compile(r"\x1b\[[0-9;]*m")


def read_text(path: Path) -> str:
    if not path.exists():
        return ""
    return ANSI_RE.sub("", path.read_text(errors="ignore"))


def load_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def to_int(value, default=None):
    try:
        if value in (None, "", "auto"):
            return default
        return int(value)
    except Exception:
        return default


def short_cpu_list(cpu_ids: str, max_items: int = 8) -> str:
    if not cpu_ids:
        return ""
    parts = [p.strip() for p in cpu_ids.split(",") if p.strip()]
    if len(parts) <= max_items * 2:
        return ",".join(parts)
    return ",".join(parts[:max_items]) + " ... " + ",".join(parts[-max_items:])


def parse_run_log(text: str) -> dict:
    router = re.search(
        r"HYBRID-ROUTER-INIT.*?strategy=(\S+) priority=(\S+) "
        r"cpu_max_num_seqs=(\d+) num_cpu_engines=(\d+).*?gate=([^ ]+)",
        text,
    )
    dispatch = re.findall(
        r"HYBRID-ROUTER-DISPATCH.*?n=(\d+).*?cpu_count=(\d+) "
        r"gpu_count=(\d+) cpu_in_flight=(\d+) gpu_in_flight=(\d+)",
        text,
    )
    waves = re.findall(
        r"HYBRID-WAVE.*?engine=(\d+) wave closed "
        r"\(accepted=(\d+), batch_size=(\d+)\)",
        text,
    )
    drains = re.findall(
        r"HYBRID-WAVE.*?engine=(\d+) wave drained \(accepted=(\d+)\)",
        text,
    )
    stats = re.findall(
        r"HYBRID-ROUTER-STATS.*?GPU=.*?\((\d+) reqs\), "
        r"CPU=.*?\((\d+) reqs\), in_flight_cpu=(\d+)/(\d+)",
        text,
    )

    last_dispatch = dispatch[-1] if dispatch else None
    last_stats = stats[-1] if stats else None
    return {
        "router_strategy_log": router.group(1) if router else None,
        "router_priority_log": router.group(2) if router else None,
        "cpu_max_num_seqs_log": to_int(router.group(3)) if router else None,
        "num_cpu_engines_log": to_int(router.group(4)) if router else None,
        "router_gate": router.group(5) if router else None,
        "dispatch_n": to_int(last_dispatch[0]) if last_dispatch else None,
        "dispatch_cpu_count": to_int(last_dispatch[1]) if last_dispatch else None,
        "dispatch_gpu_count": to_int(last_dispatch[2]) if last_dispatch else None,
        "dispatch_cpu_in_flight": to_int(last_dispatch[3]) if last_dispatch else None,
        "dispatch_gpu_in_flight": to_int(last_dispatch[4]) if last_dispatch else None,
        "wave_closed_events": len(waves),
        "wave_closed_total_accepted": sum(to_int(w[1], 0) for w in waves),
        "wave_closed_detail": "; ".join(
            f"e{e}:accepted={a},batch={b}" for e, a, b in waves
        ),
        "wave_drained_events": len(drains),
        "wave_drained_total_accepted": sum(to_int(d[1], 0) for d in drains),
        "wave_drained_detail": "; ".join(
            f"e{e}:accepted={a}" for e, a in drains
        ),
        "stats_gpu_reqs": to_int(last_stats[0]) if last_stats else None,
        "stats_cpu_reqs": to_int(last_stats[1]) if last_stats else None,
        "stats_in_flight_cpu": to_int(last_stats[2]) if last_stats else None,
        "stats_cpu_max_num_seqs": to_int(last_stats[3]) if last_stats else None,
    }


def parse_boot_log(text: str) -> dict:
    cpus = re.findall(r"local_omp_cpuid='([^']+)'", text)
    post = re.findall(
        r"post-init: torch_threads=(\d+) process_threads=(\d+) "
        r"cpu_affinity=(\d+) cores ([^\n]+)",
        text,
    )
    envs = re.findall(
        r"HYBRID-CPU-ENV.*?OMP=(\d+).*?ONEDNN_ISA=([^ ]+) "
        r"BIND=([^ ]+) sched_affinity_count=(\d+)",
        text,
    )
    return {
        "cpu_pinning_lists": " | ".join(short_cpu_list(c) for c in cpus),
        "cpu_pinning_counts": " | ".join(str(len([x for x in c.split(',') if x])) for c in cpus),
        "post_torch_threads": " | ".join(p[0] for p in post),
        "post_process_threads": " | ".join(p[1] for p in post),
        "env_omp": " | ".join(e[0] for e in envs),
        "env_onednn_isa": " | ".join(sorted(set(e[1] for e in envs))),
        "env_bind": " | ".join(sorted(set(e[2] for e in envs))),
        "sched_affinity_count": " | ".join(e[3] for e in envs),
    }


## Load Results

In [ ]:
rows = []

for d in sorted(p for p in ROOT.iterdir() if p.is_dir()):
    result_path = d / "gpu_only.json"
    mode = "gpu_only" if result_path.exists() else "hybrid"
    if mode == "hybrid":
        result_path = d / "hybrid.json"
    if not result_path.exists():
        continue

    result = load_json(result_path)
    system = load_json(d / "system_info.json")
    hybrid_cfg = system.get("hybrid_config", {})
    bench_cfg = system.get("benchmark_config", {})

    run_info = parse_run_log(read_text(d / "hybrid_server_run.log"))
    boot_info = parse_boot_log(read_text(d / "hybrid_server_boot.log"))

    row = {
        "run_dir": d.name,
        "mode": mode,
        "date": result.get("date"),
        "model_id": result.get("model_id"),
        "num_prompts": result.get("num_prompts"),
        "completed": result.get("completed"),
        "duration_s": result.get("duration"),
        "wall_time_s": result.get("wall_time_s"),
        "request_throughput": result.get("request_throughput"),
        "output_throughput": result.get("output_throughput"),
        "total_token_throughput": result.get("total_token_throughput"),
        "mean_ttft_ms": result.get("mean_ttft_ms"),
        "p99_ttft_ms": result.get("p99_ttft_ms"),
        "mean_tpot_ms": result.get("mean_tpot_ms"),
        "p99_tpot_ms": result.get("p99_tpot_ms"),
        "routing_strategy_cfg": hybrid_cfg.get("routing_strategy"),
        "routing_priority_cfg": hybrid_cfg.get("routing_priority"),
        "cpu_max_seqs_cfg": to_int(hybrid_cfg.get("cpu_max_seqs")),
        "cpu_threads_cfg": to_int(hybrid_cfg.get("cpu_threads")),
        "num_cpu_engines_cfg": to_int(hybrid_cfg.get("num_cpu_engines")),
        "numa_aware_cfg": hybrid_cfg.get("numa_aware"),
        "input_len": to_int(bench_cfg.get("input_len")),
        "output_len": to_int(bench_cfg.get("output_len")),
        "request_rate": bench_cfg.get("request_rate") or result.get("request_rate"),
    }
    row.update(run_info)
    row.update(boot_info)
    rows.append(row)

df = pd.DataFrame(rows).sort_values(["mode", "date", "run_dir"]).reset_index(drop=True)

gpu = df[df["mode"] == "gpu_only"].sort_values("date").iloc[-1]
gpu_wall = float(gpu["wall_time_s"])
gpu_duration = float(gpu["duration_s"])
gpu_req_tps = float(gpu["request_throughput"])
gpu_out_tps = float(gpu["output_throughput"])

df["wall_vs_gpu_x"] = df["wall_time_s"] / gpu_wall
df["duration_vs_gpu_x"] = df["duration_s"] / gpu_duration
df["request_tput_vs_gpu"] = df["request_throughput"] / gpu_req_tps
df["output_tput_vs_gpu"] = df["output_throughput"] / gpu_out_tps
df["estimated_cpu_tail_s"] = df["wall_time_s"] - gpu_wall
df.loc[df["mode"] == "gpu_only", "estimated_cpu_tail_s"] = 0.0

df


## Executive Summary

In [ ]:
hy = df[df["mode"] == "hybrid"].copy()

summary_lines = [
    f"- GPU-only baseline: wall={gpu_wall:.2f}s, duration={gpu_duration:.2f}s, request throughput={gpu_req_tps:.2f} req/s, output throughput={gpu_out_tps:.0f} tok/s.",
]

if not hy.empty:
    best_wall = hy.loc[hy["wall_time_s"].idxmin()]
    worst_wall = hy.loc[hy["wall_time_s"].idxmax()]
    summary_lines.extend([
        f"- Best hybrid wall: {best_wall['run_dir']} wall={best_wall['wall_time_s']:.2f}s ({best_wall['wall_vs_gpu_x']:.1f}x GPU-only).",
        f"- Worst hybrid wall: {worst_wall['run_dir']} wall={worst_wall['wall_time_s']:.2f}s ({worst_wall['wall_vs_gpu_x']:.1f}x GPU-only).",
        "- Hybrid runs still show the same pattern: GPU bulk finishes quickly, then CPU wave drain dominates total wall time.",
        "- The visible changed dimension in the 20260415 runs is CPU pinning range; routing/wave behavior remains the same shape as prior analysis.",
    ])

display(Markdown("\n".join(summary_lines)))


## Result Table: GPU-only Baseline vs Hybrid

In [ ]:
cols = [
    "run_dir", "mode", "wall_time_s", "duration_s", "wall_vs_gpu_x",
    "request_throughput", "request_tput_vs_gpu", "output_throughput",
    "output_tput_vs_gpu", "mean_ttft_ms", "mean_tpot_ms",
    "cpu_max_seqs_cfg", "cpu_threads_cfg", "num_cpu_engines_cfg",
    "dispatch_cpu_count", "dispatch_gpu_count", "estimated_cpu_tail_s",
]

display(
    df[cols]
    .sort_values(["mode", "cpu_max_seqs_cfg", "cpu_threads_cfg", "wall_time_s"], na_position="first")
    .style.format({
        "wall_time_s": "{:.2f}",
        "duration_s": "{:.2f}",
        "wall_vs_gpu_x": "{:.1f}x",
        "request_throughput": "{:.3f}",
        "request_tput_vs_gpu": "{:.4f}",
        "output_throughput": "{:.1f}",
        "output_tput_vs_gpu": "{:.4f}",
        "mean_ttft_ms": "{:.1f}",
        "mean_tpot_ms": "{:.1f}",
        "estimated_cpu_tail_s": "{:.1f}",
    })
)


## Routing / Wave Evidence from Logs

In [ ]:
routing_cols = [
    "run_dir", "routing_strategy_cfg", "routing_priority_cfg",
    "cpu_max_seqs_cfg", "cpu_threads_cfg", "num_cpu_engines_cfg",
    "cpu_max_num_seqs_log", "num_cpu_engines_log", "router_gate",
    "dispatch_n", "dispatch_cpu_count", "dispatch_gpu_count",
    "wave_closed_events", "wave_closed_total_accepted", "wave_closed_detail",
    "wave_drained_events", "wave_drained_total_accepted", "wave_drained_detail",
]

display(df[df["mode"] == "hybrid"][routing_cols].sort_values("run_dir"))


## CPU Pinning Evidence from Boot Logs

이 표는 서버 부팅 로그에 직접 찍힌 `local_omp_cpuid`만 사용합니다. topology 해석이나 `inspect.txt`는 사용하지 않습니다.

In [ ]:
pin_cols = [
    "run_dir", "cpu_max_seqs_cfg", "cpu_threads_cfg", "num_cpu_engines_cfg",
    "env_omp", "post_torch_threads", "sched_affinity_count",
    "env_onednn_isa", "env_bind", "cpu_pinning_counts", "cpu_pinning_lists",
]

display(df[df["mode"] == "hybrid"][pin_cols].sort_values("run_dir"))


## Plots

In [ ]:
if plt is None:
    display(Markdown("`matplotlib` is not installed in this environment, so plots are skipped. The tables above contain the same data."))
else:
    plot_df = df.copy()
    plot_df["label"] = plot_df.apply(
        lambda r: "GPU-only" if r["mode"] == "gpu_only" else f"seq={int(r['cpu_max_seqs_cfg'])},thr={int(r['cpu_threads_cfg'])}",
        axis=1,
    )

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

    axes[0].bar(plot_df["label"], plot_df["wall_time_s"])
    axes[0].set_title("Wall time (s)")
    axes[0].set_ylabel("seconds")
    axes[0].tick_params(axis="x", rotation=35)

    axes[1].bar(plot_df["label"], plot_df["wall_vs_gpu_x"])
    axes[1].set_title("Wall slowdown vs GPU-only")
    axes[1].set_ylabel("x GPU-only")
    axes[1].tick_params(axis="x", rotation=35)

    hy_plot = plot_df[plot_df["mode"] == "hybrid"]
    x = range(len(hy_plot))
    axes[2].bar(x, hy_plot["dispatch_gpu_count"], label="GPU requests")
    axes[2].bar(x, hy_plot["dispatch_cpu_count"], bottom=hy_plot["dispatch_gpu_count"], label="CPU requests")
    axes[2].set_xticks(list(x), hy_plot["label"], rotation=35)
    axes[2].set_title("Final dispatch split")
    axes[2].set_ylabel("requests")
    axes[2].legend()

    plt.tight_layout()
    plt.show()


## Grouped Comparison by `cpu_max_num_seqs` and `cpu_threads`

In [ ]:
group_cols = ["cpu_max_seqs_cfg", "cpu_threads_cfg"]
grouped = (
    df[df["mode"] == "hybrid"]
    .groupby(group_cols, dropna=False)
    .agg(
        runs=("run_dir", "count"),
        wall_mean_s=("wall_time_s", "mean"),
        wall_min_s=("wall_time_s", "min"),
        wall_max_s=("wall_time_s", "max"),
        wall_vs_gpu_mean_x=("wall_vs_gpu_x", "mean"),
        request_tput_mean=("request_throughput", "mean"),
        output_tput_mean=("output_throughput", "mean"),
        dispatch_cpu_count=("dispatch_cpu_count", "max"),
        dispatch_gpu_count=("dispatch_gpu_count", "max"),
        wave_total_accepted=("wave_closed_total_accepted", "max"),
    )
    .reset_index()
)

display(grouped.style.format({
    "wall_mean_s": "{:.2f}",
    "wall_min_s": "{:.2f}",
    "wall_max_s": "{:.2f}",
    "wall_vs_gpu_mean_x": "{:.1f}x",
    "request_tput_mean": "{:.3f}",
    "output_tput_mean": "{:.1f}",
}))


## Mechanical Conclusion

아래 셀은 현재 결과로부터 기계적으로 결론 문장을 생성합니다.

In [ ]:
seq1 = hy[hy["cpu_max_seqs_cfg"] == 1]
seq16 = hy[hy["cpu_max_seqs_cfg"] == 16]

lines = []
if not seq1.empty:
    lines.append(
        f"- `cpu_max_num_seqs=1`: CPU dispatch={int(seq1['dispatch_cpu_count'].max())} requests, "
        f"wall range={seq1['wall_time_s'].min():.2f}~{seq1['wall_time_s'].max():.2f}s "
        f"({seq1['wall_vs_gpu_x'].min():.1f}~{seq1['wall_vs_gpu_x'].max():.1f}x GPU-only)."
    )
if not seq16.empty:
    lines.append(
        f"- `cpu_max_num_seqs=16`: CPU dispatch={int(seq16['dispatch_cpu_count'].max())} requests, "
        f"wall range={seq16['wall_time_s'].min():.2f}~{seq16['wall_time_s'].max():.2f}s "
        f"({seq16['wall_vs_gpu_x'].min():.1f}~{seq16['wall_vs_gpu_x'].max():.1f}x GPU-only)."
    )

lines.extend([
    "- GPU-only 대비 hybrid는 모든 현재 H100x8 하위 결과에서 느립니다.",
    "- `cpu_max_num_seqs=16`은 CPU가 가져가는 request 수를 32개로 늘리지만, wall time을 줄이지 않고 긴 CPU tail을 만듭니다.",
    "- 20260415 결과의 pinning 범위는 boot log 기준으로 `0..55` / `56..111` 계열입니다. 이전 분석의 `112..167` / `168..223` 계열과 pinning은 다르지만, wave/routing/tail 패턴은 동일합니다.",
])

display(Markdown("\n".join(lines)))
